# ДЗ1 — Иван Башкин: запуск, тесты и замеры

Дата запуска: 2026-11-20. Ячейки выполняют сценарий последовательно из корня репозитория. Все фактические показатели должны появиться только после выполнения соответствующих шагов.

## Шаги
0. Проверка окружения и `.env`  
1. Зависимости  
2. PostgreSQL  
3. Миграции  
4. Малый набор и счётчики  
5. Сервис малого набора  
6. Тесты  
7. Benchmark малого набора  
8. Разложение времени малого набора  
9. Остановка сервиса  
10. Рабочий набор и счётчики  
11. Сервис и benchmark рабочего набора  
12. Остановка и итоговая сводка

## Шаг 0. Окружение
Показываются версии Python, Docker/Compose, `psql` (если установлен) и pip. Если `.env` отсутствует, он создаётся из `.env.example`; содержимое и секреты не выводятся.

In [1]:
%%bash
cd ..
set -e
python3 --version
docker --version
docker compose version
if command -v psql >/dev/null 2>&1; then psql --version; else echo 'psql: не установлен'; fi
python3 -m pip --version
if [ -f .env ]; then echo '.env уже существует; оставлен без изменений'; else cp .env.example .env && echo '.env создан из .env.example'; fi

Python 3.14.10


Docker version 29.9.3, build 5457199aa5


Docker Compose version 5.7.1


psql (PostgreSQL) 18.7


pip 26.2.2 from /usr/lib/python3.14/site-packages/pip (python 3.14)


.env уже существует; оставлен без изменений


## Шаг 1. Зависимости
Создаётся виртуальное окружение, только если его ещё нет. Зависимости устанавливаются явным pip из `.venv`.

In [2]:
%%bash
cd ..
set -e
if [ ! -x .venv/bin/python ]; then python3 -m venv .venv; fi
.venv/bin/python --version
.venv/bin/pip install -r requirements.txt

Python 3.14.10


requirements.txt (line 1)) (0.165.0)


(from -r requirements.txt (line 2)) (0.0.34)


orn[standard]<1.0,>=0.30->-r requirements.txt (line 3)) (0.54.0)

lchemy<3.0,>=2.0 in ./.venv/lib/python3.14/site-packages (from -r requirements.txt (line 4)) (2.1.9)

om -r requirements.txt (line 5)) (2.9.13)

... (33 outputs left out)


[notice] A new release of pip is available: 2

6.2.2 -> 26.3
[notice] To upda

te, run: /home/ewvn/MISIS/OKSP/kr1/ivan_bashkin/.venv/bin/python3 -m pip install --upgra

de pip


## Шаг 2. База данных
Запускается Compose и PostgreSQL. Проверка готовности повторяется с паузой, не более 60 секунд; успешная проверка `pg_isready` будет видна в выводе.

In [3]:
%%bash
cd ..
set -e
docker compose up -d
ready=0
for attempt in $(seq 1 30); do
  echo "Проверка PostgreSQL: попытка ${attempt}/30"
  if docker compose exec -T ivan_bashkin_postgres pg_isready -U app -d ivan_bashkin; then ready=1; break; fi
  if [ "$attempt" -lt 30 ]; then sleep 2; fi
done
if [ "$ready" -ne 1 ]; then echo 'PostgreSQL не готов за 60 секунд' >&2; exit 1; fi

 Container ivan_bashkin_postgres Running 


Проверка PostgreSQL: попытка 1/30


/var/run/postgresql:5432 - accepting connections


## Шаг 3. Миграции
Применяются миграции, затем выводится фактическая текущая ревизия Alembic.

In [4]:
%%bash
cd ..
set -e
.venv/bin/alembic upgrade head
.venv/bin/alembic current

INFO  [alembic.runtime.migration] Context impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Con

text impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Will assume transactional DDL.
INFO  [al

embic.runtime.migration] Will assume transactional DDL.


INFO  [alembic.runtime.migration] Context impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Con

text impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Will assume transactional DDL.


INFO  [alembic.runtime.migration] Will assume transactional DDL.


0001_initial (head)


> **Зафиксированная ошибка первого выполнения:** `alembic upgrade head` завершился `ModuleNotFoundError: No module named 'psycopg'`. URL `postgresql://` выбирал драйвер psycopg, тогда как проект устанавливает `psycopg2-binary`. URL нормализован к psycopg2 в `app/database.py` и `migrations/env.py`; после исправления шаг выполняется повторно.

## Шаг 4. Малый набор и подсчёт строк
Seed заполняет малый набор. Следующая Python-ячейка считает строки непосредственно в БД для всех ORM-таблиц и печатает таблицу.

In [5]:
%%bash
cd ..
set -euo pipefail
.venv/bin/python seed.py --type=small | sed -E 's#(Login: admin / ).*#\1[REDACTED]#'

Seeded 50 readers, 100 books, 200 copies, 300 loans. Login: admin / [REDACTED]


### Подсчёт малого набора
Значения ниже получены запросами `COUNT(*)` к таблицам базы, а не взяты из сообщения seed.

In [6]:
import json
import subprocess

count_script = '''
import json
from sqlalchemy import func, select
from app.database import SessionLocal
from app.models import Book, BookCopy, Loan, Reader, User
models = [("users", User), ("readers", Reader), ("books", Book), ("book_copies", BookCopy), ("loans", Loan)]
with SessionLocal() as db:
    counts = {table: db.scalar(select(func.count()).select_from(model)) for table, model in models}
print(json.dumps(counts))
'''
count_result = subprocess.run([".venv/bin/python", "-c", count_script], cwd="..", check=True, capture_output=True, text=True)
counts_small = json.loads(count_result.stdout)
print("Объём: small")
print("| Таблица | Фактическое число строк |\n|---|---:|")
for table, count in counts_small.items():
    print(f"| {table} | {count} |")

Объём: small
| Таблица | Фактическое число строк |
|---|---:|
| users | 1 |
| readers | 50 |
| books | 100 |
| book_copies | 200 |
| loans | 300 |


> **Зафиксированная ошибка второго выполнения:** Python-ядро ноутбука работает из каталога `notebooks/`, поэтому относительный путь `.venv/bin/python` не найден. Ячейки подсчёта используют `cwd=".."` при запуске Python из виртуального окружения; это сохраняет корень проекта как рабочий каталог скрипта.

## Шаг 5. Сервис малого набора
Uvicorn запускается в фоне, лог и PID сохраняются в `logs/`. Ожидание `/docs` ограничено 30 секундами.

In [7]:
%%bash
cd ..
set -e
mkdir -p logs
nohup .venv/bin/uvicorn app.main:app --port 8000 > logs/uvicorn_small.log 2>&1 &
echo $! > logs/uvicorn_small.pid
ready=0
for attempt in $(seq 1 30); do
  echo "Ожидание API /docs: попытка ${attempt}/30"
  if curl --fail --silent --show-error --max-time 2 http://localhost:8000/docs >/dev/null; then ready=1; break; fi
  sleep 1
done
if [ "$ready" -ne 1 ]; then echo 'API не ответил за 30 секунд' >&2; cat logs/uvicorn_small.log; kill "$(cat logs/uvicorn_small.pid)" 2>/dev/null || true; exit 1; fi
echo "Uvicorn PID: $(cat logs/uvicorn_small.pid)"

Ожидание API /docs: попытка 1/30


curl: (7) Failed to connect to localhost:8000 after 0 ms: Could not connect to server


Ожидание API /docs: попытка 2/30


Uvicorn PID: 21133


## Шаг 6. Тесты
Запускается `pytest -v` на текущей тестовой конфигурации. Полный вывод сохраняется в `logs/pytest.log`; при ошибке ячейка завершится с ошибкой и попытается остановить сервис малого набора.

In [8]:
%%bash
cd ..
set -o pipefail
mkdir -p logs
.venv/bin/pytest -v 2>&1 | tee logs/pytest.log || { status=$?; if [ -f logs/uvicorn_small.pid ]; then kill "$(cat logs/uvicorn_small.pid)" 2>/dev/null || true; fi; exit "$status"; }

============================= test session starts ==============================
platform li

nux -- Python 3.14.10, pytest-8.4.2, pluggy-1.7.0 -- /home/ewvn/MISIS/OKSP/kr1/ivan_bashkin/.venv/bi

n/python3
cachedir: .pytest_cache
rootdir: /home/ewvn/MISIS/OKSP/kr1/ivan_bashkin
configfile: pytest

.ini
plugins: anyio-4.16.0
collecting ... 

collected 22 items

tests/test_api.py::test_protected_routes_require_authentication 

PASSED   [  4%]

... (45 outputs left out)

`starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import

 TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warning

s.html
======================== 22 passed, 1 warning in 3.95s[3

3m =========================


## Шаг 7. Замеры малого набора
Выполняются 30 измерений после 5 прогревочных запросов. Скрипту передаётся метка `small`; CSV сохраняется отдельным файлом, затем выводится полностью.

In [9]:
%%bash
cd ..
set -o pipefail
.venv/bin/python benchmark.py --requests 30 --warmup 5 --url http://localhost:8000 --dataset small --csv benchmark_results_small.csv || { status=$?; if [ -f logs/uvicorn_small.pid ]; then kill "$(cat logs/uvicorn_small.pid)" 2>/dev/null || true; fi; exit "$status"; }
test -f benchmark_results_small.csv
cat benchmark_results_small.csv

Операция                               Параметры, Б    p50, ms    p95, ms    max, 

ms   БД p50, ms  код p50, ms   SQL p50
POST /api/v1/auth/login                          32     

104.56     106.96     109.02         0.83       101.47         1
GET /api/v1/loans                  

              14       8.10       9.19       9.47         1.92         4.14         3
GET /api/v1/lo

ans/{id}                            1       7.64       7.95       7.99         1.84         3.79    

     2
GET /api/v1/summary                               0       9.85      10.53      10.62         

... (8 outputs left out)

mmary,0,30,5,9.850629000538902,10.525542999857862,10.620923999340448,3.2725,4.7675,4
small,POST /ap

i/v1/loans/{id}/return,3,30,5,14.767535999908432,15.295088000129908,15.455529999599094,3.81299999999

99997,8.782,6
small,POST /api/v1/loans,28,30,5,16.98186949988667,18.716722000135633,22.386689000086

335,4.553,10.211500000000001,7


## Шаг 8. Разложение времени малого набора
Отдельного `breakdown.py` нет. Эквивалентное разложение уже входит в `benchmark.py`: ниже из CSV выводятся ответ, время БД, время кода и число SQL-запросов для списка и создания выдачи.

In [10]:
import csv

with open("../benchmark_results_small.csv", newline="", encoding="utf-8") as source:
    rows = list(csv.DictReader(source))
print("| Операция | Ответ p50, ms | БД p50, ms | Код p50, ms | SQL-запросы p50 |\n|---|---:|---:|---:|---:|")
for row in rows:
    if row["operation"] in {"GET /api/v1/loans", "POST /api/v1/loans"}:
        print(f"| {row['operation']} | {row['p50_ms']} | {row['db_p50_ms']} | {row['code_p50_ms']} | {row['db_queries_p50']} |")

| Операция | Ответ p50, ms | БД p50, ms | Код p50, ms | SQL-запросы p50 |
|---|---:|---:|---:|---:|
| GET /api/v1/loans | 8.097838999674423 | 1.9209999999999998 | 4.1370000000000005 | 3 |
| POST /api/v1/loans | 16.98186949988667 | 4.553 | 10.211500000000001 | 7 |


## Шаг 9. Остановка сервиса малого набора
Сервис останавливается по записанному PID; затем проверяется, что HTTP-порт больше не отвечает.

In [11]:
%%bash
cd ..
set -e
pid=$(cat logs/uvicorn_small.pid)
kill "$pid"
stopped=0
for attempt in $(seq 1 10); do
  if ! curl --fail --silent --max-time 1 http://localhost:8000/docs >/dev/null; then stopped=1; break; fi
  sleep 1
done
if [ "$stopped" -ne 1 ]; then echo 'Порт 8000 всё ещё отвечает' >&2; exit 1; fi
echo "Процесс $pid остановлен; curl к /docs завершился ошибкой, порт не отвечает."

Процесс 21133 остановлен; curl к /docs завершился ошибкой, пор

т не отвечает.


## Шаг 10. Рабочий набор и подсчёт строк
Загружается `large`; Python-ячейка независимо от сообщения seed считает все строки в базе.

In [12]:
%%bash
cd ..
set -euo pipefail
.venv/bin/python seed.py --type=large | sed -E 's#(Login: admin / ).*#\1[REDACTED]#'

Seeded 3000 readers, 10000 books, 30000 copies, 75000 loans. Login: admin / [REDACTED]


### Подсчёт рабочего набора
Значения ниже получены запросами `COUNT(*)` к таблицам базы, а не взяты из сообщения seed.

In [13]:
count_result = subprocess.run([".venv/bin/python", "-c", count_script], cwd="..", check=True, capture_output=True, text=True)
counts_large = json.loads(count_result.stdout)
print("Объём: large")
print("| Таблица | Фактическое число строк |\n|---|---:|")
for table, count in counts_large.items():
    print(f"| {table} | {count} |")

Объём: large
| Таблица | Фактическое число строк |
|---|---:|
| users | 1 |
| readers | 3000 |
| books | 10000 |
| book_copies | 30000 |
| loans | 75000 |


## Шаг 11. Сервис и замеры рабочего набора
Uvicorn запускается с отдельным логом, готовность API проверяется до 30 секунд. Затем выполняются замеры `large` и CSV сохраняется отдельно от малого набора.

In [14]:
%%bash
cd ..
set -e
nohup .venv/bin/uvicorn app.main:app --port 8000 > logs/uvicorn_large.log 2>&1 &
echo $! > logs/uvicorn_large.pid
ready=0
for attempt in $(seq 1 30); do
  echo "Ожидание API /docs: попытка ${attempt}/30"
  if curl --fail --silent --show-error --max-time 2 http://localhost:8000/docs >/dev/null; then ready=1; break; fi
  sleep 1
done
if [ "$ready" -ne 1 ]; then echo 'API не ответил за 30 секунд' >&2; cat logs/uvicorn_large.log; kill "$(cat logs/uvicorn_large.pid)" 2>/dev/null || true; exit 1; fi
.venv/bin/python benchmark.py --requests 30 --warmup 5 --url http://localhost:8000 --dataset large --csv benchmark_results_large.csv || { status=$?; kill "$(cat logs/uvicorn_large.pid)" 2>/dev/null || true; exit "$status"; }
test -f benchmark_results_large.csv
cat benchmark_results_large.csv

Ожидание API /docs: попытка 1/30


curl: (7) Failed to connect to localhost:8000 after 0 ms: Could not connect to server


Ожидание API /docs: попытка 2/30


Операция                               Параметры, Б    p50, ms    p95, ms    max, 

ms   БД p50, ms  код p50, ms   SQL p50
POST /api/v1/auth/login                          32     

105.32     108.77     109.81         0.82       102.31         1
GET /api/v1/loans                  

... (11 outputs left out)

,5,97.24563850022605,104.09874700053479,104.10585799945693,89.959,5.2425,4
large,POST /api/v1/loans

/{id}/return,5,30,5,14.467395500560087,15.885314000115613,21.505941999748757,3.85,8.747499999999999,

6
large,POST /api/v1/loans,28,30,5,24.927538000156346,32.61785399990913,32.86776799996005,12.7105,1

0.334,7


### Разложение времени рабочего набора
Из фактического CSV показываются время ответа, SQL-время, время кода и количество запросов для `GET /api/v1/loans` и `POST /api/v1/loans`.

In [15]:
with open("../benchmark_results_large.csv", newline="", encoding="utf-8") as source:
    rows_large = list(csv.DictReader(source))
print("| Операция | Ответ p50, ms | БД p50, ms | Код p50, ms | SQL-запросы p50 |\n|---|---:|---:|---:|---:|")
for row in rows_large:
    if row["operation"] in {"GET /api/v1/loans", "POST /api/v1/loans"}:
        print(f"| {row['operation']} | {row['p50_ms']} | {row['db_p50_ms']} | {row['code_p50_ms']} | {row['db_queries_p50']} |")

| Операция | Ответ p50, ms | БД p50, ms | Код p50, ms | SQL-запросы p50 |
|---|---:|---:|---:|---:|
| GET /api/v1/loans | 17.702132500289736 | 11.046 | 4.6080000000000005 | 3 |
| POST /api/v1/loans | 24.927538000156346 | 12.7105 | 10.334 | 7 |


## Шаг 12. Остановка и итог
Останавливается сервис большого набора, проверяется отсутствие ответа на порту; `docker compose ps` сохраняется и выводится. PostgreSQL не останавливается.

In [16]:
%%bash
cd ..
set -e
pid=$(cat logs/uvicorn_large.pid)
kill "$pid"
stopped=0
for attempt in $(seq 1 10); do
  if ! curl --fail --silent --max-time 1 http://localhost:8000/docs >/dev/null; then stopped=1; break; fi
  sleep 1
done
if [ "$stopped" -ne 1 ]; then echo 'Порт 8000 всё ещё отвечает' >&2; exit 1; fi
docker compose ps | tee logs/docker-compose-ps.log

NAME                    IMAGE         COMMAND                  SERVICE                 CREATED      

   STATUS                   PORTS
ivan_bashkin_postgres   postgres:16   "docker-entrypoint.s…"   i

van_bashkin_postgres   6 minutes ago   Up 6 minutes (healthy)   0.0.0.0:5432->5432/tcp, [::]:5432->5

432/tcp


## Итоговая сводка
Печатаются фактические счётчики из Python-ячеек, итоговая строка pytest (если тесты завершились), а также пути к CSV и логам. Отсутствующие результаты обозначаются как «не выполнено».

In [17]:
from pathlib import Path

print("Счётчики строк (small):", counts_small if "counts_small" in globals() else "не выполнено")
print("Счётчики строк (large):", counts_large if "counts_large" in globals() else "не выполнено")
pytest_log = Path("../logs/pytest.log")
if pytest_log.exists():
    summaries = [line.strip() for line in pytest_log.read_text(encoding="utf-8", errors="replace").splitlines() if " passed" in line or " failed" in line or "error" in line.lower()]
    print("Итог pytest:", summaries[-1] if summaries else "строка итога не найдена в полном логе")
else:
    print("Итог pytest: не выполнено")
for name in ["benchmark_results_small.csv", "benchmark_results_large.csv", "logs/pytest.log", "logs/uvicorn_small.log", "logs/uvicorn_large.log", "logs/docker-compose-ps.log"]:
    print(f"{name}: {'создан' if Path('../' + name).exists() else 'не выполнено'}")

Счётчики строк (small): {'users': 1, 'readers': 50, 'books': 100, 'book_copies': 200, 'loans': 300}
Счётчики строк (large): {'users': 1, 'readers': 3000, 'books': 10000, 'book_copies': 30000, 'loans': 75000}
Итог pytest: ======================== 22 passed, 1 warning in 3.95s =========================
benchmark_results_small.csv: создан
benchmark_results_large.csv: создан
logs/pytest.log: создан
logs/uvicorn_small.log: создан
logs/uvicorn_large.log: создан
logs/docker-compose-ps.log: создан
